# LLM-based Meeting-Video Analysis - Prototype
This notebook includes a prototype implementation of the LLM-based Meeting-Video Analysis. By following the described steps, a report is generated and exported as markdown file.

This notebook requires a transcript provided as text. The transcript generation was excluded from this notebook to reduce complexity.
The original prototype which handles the transcript generation was excluded from the publication, as it would reveal the authors due to its domain.

## LLM Configuration
Provide an OpenAI key to run the analysis.

In [1]:
# TODO: Insert API key.
OPENAI_API_KEY = "API_KEY"
OPENAI_MODEL = "gpt-4o"

In [2]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

## LLM Prompts
This section defines the individual prompts used to analyze the meetings.

In [20]:
import json

# The system prompt is added to all analysis steps to provide relevant context.
SYSTEM_PROMPT = f"""
    You are an assistant that supports a group of users in decision-making by taking a neutral position.
    Based on a given transcription of a group meeting, try to indentify the following aspects based on it.
    If information cannot be derived from the transcript, mention "This information is not included!"
    """

USER_PROMPT = f"""
Consider the following meeting transcript:

"""

def get_user_prompt(transcript, insight, alternatives=[]):
    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    if len(alternatives) > 0:
        prompt += f"\n\nThe following alternatives are considered for decision-making in the discussion: {', '.join(alternatives)}"
    return prompt

In [5]:
# Evaluate the meeting aim
def aim_prompt(transcript):
    insight = {
        "prompt": "What is the aim of the decision-making process? Start the description with 'The aim of the meeting is to...'. Explain why this is the aim and how you identified it.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "aim": string,
            "explanation": string
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [6]:
# Identify discussed alternatives
def alternatives_prompt(transcript):
    insight = {
        "prompt": "Which decision alternatives can you identify from the transcript? Provide a complete list of mentioned alternatives.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "alternatives": []
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

# Recommend additional alternatives
def recommended_alternatives_prompt(transcript, alternatives=[]):
    insight = {
        "prompt": "Suggest further relevant decision alternatives that have not been covered in the discussion. Align their style with the given alternatives. Explain why you consider them relevant.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            alternatives": [
                {
                    "alternative": string,
                    "explanation": string
                },
                ...
            ]
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    if len(alternatives) > 0:
        prompt += f"\n\nThe following alternatives are considered for decision-making in the discussion: {', '.join(alternatives)}"
        prompt += "\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [7]:
# Identify discussed arguments
def argument_prompt(transcript, alternative):
    insight = {
        "prompt": "Identify all discussed arguments for and against it. Explain them such that they can be understood without the context of the transcript. Mention the reference phrase in the transcript. Afterwards, identify all arguments for and against considering aspects that have not been covered in the discussion.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "arguments_for": [
                {
                    "argument": string,
                    "reference": string
                },
                ...
            ],
            "arguments_against": [
                {
                    "argument": string,
                    "reference": string
                },
                ...
            ],
            "additional_arguments_for": [
                {
                    "argument": string
                },
                ...
            ],
            "additional_arguments_against": [
                {
                    "argument": string
                },
                ...
            ]
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"\n\nConsider the following decision alternative in the discussion: {alternative}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [8]:
# Identify the grouping using in the discussion, if available
def groups_only_prompt(transcript, alternatives=[]):
    insight = {
        "prompt": "Analyze the grouping of alternatives used in the discussion. It could be a single list or multiple groups. From the transcript, identify and label these groups.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "groups": [
                {
                    "label": string,
                },
                ...
            ]
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    if len(alternatives) > 0:
        prompt += f"\n\nThe following alternatives are considered for decision-making in the discussion: {', '.join(alternatives)}"
        prompt += "\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

# Assign an alternative to one of the identified groups
def groups_assignment_only_single(transcript, alternative, groups=[]):
    insight = {
        "prompt": "Assign the alternative to one of the available groups based on the discussion.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "group": string
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"\n\nConsider the following decision alternative in the discussion: {alternative}\n\n"
    if len(groups) > 0:
        prompt += f"\n\nThe following groups are available: {', '.join(groups)}"
        prompt += "\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [9]:
# Identify the discussed ranking within a group
def single_group_ranking_prompt(transcript, group, alternatives):
    insight = {
        "prompt": "Identify the ranking of alternatives within the group based on the discussion. Rank all alternatives and assign rank -1 to any unranked options. Explain the reasons for the individual rankings.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "ranking": [
                {
                        "alternative": string,
                        "rank": number,
                        "explanation": string
                },
                ...
            ]
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"\n\nConsider the group '{group}' and its assigned alternatives: {', '.join(alternatives)}\n"
    prompt += "\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [10]:
# Identify relationships between groups
def relationships_prompt(transcript, alternatives=[]):
    insight = {
        "prompt": "Which interrelationships exist between the alternatives? Consider the following types of relationships and identify all of them:\n- Dependencies, i.e., one alternative is a prequisite for another. \n- Redundancies, i.e., one alternative is very similar to another.\nExplain your reasoning for each relationship. If there are no dependencies, leave the results empty.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "dependencies": [
                {
                    "description": string,
                    "explanation": string
                },
                ...
            ],
            "redundancies": [
                {
                    "description": string,
                    "explanation": string
                },
                ...
            ]
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    if len(alternatives) > 0:
        prompt += f"\n\nThe following alternatives are considered for decision-making in the discussion: {', '.join(alternatives)}"
        prompt += "\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [54]:
# Evaluate the prioritization strategy
def prioritization_prompt(transcript):
    insight = {
        "prompt": "On which prioritization strategies did the group agree? Explain those strategies and how you identified them. Furthermore, suggest potential ways to improve the prioritization strategy.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "prioritization": string,
            "explanation": string,
            "improvement_suggestion": string
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

def process_evaluation_prompt(transcript):
    insight = {
        "prompt": "How would you rate the decision-making process considering the transcript? Rate it from 1 to 5, where 1 is the lowest and 5 is the highest. Explain the reason for your rating. Furthermore, suggest how it could be improved and explain why.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "rating": number,
            "explanation": string,
            "improvement_suggestion": string,
            "improvement_explanation": string
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [12]:
# Evaluate discussion fairness
def discussion_fairness_prompt(transcript):
    insight = {
        "prompt": "Analyze whether the decision-making process and discussion were fair or not. Consider the following fairness aspects:\n-Equal Participation: All participants have opportunities to meaningfully participate in the decision-making process.\n- Consideration of Opinions: All opinions are considered during decision-making.\n- Transparency: Meeting goals and topics are clearly defined.\n- Clarification and Extension: Participants can ask clarifying questions and extend aspects.\n- Reasoned Decisions: Clear explanations are provided for decisions made.\n- Respectful Communication: Absence of interruptions, dominance, or dismissive behavior.\n- Feedback and Improvement: Participants can express concerns and provide suggestions.\nList your analysis results and explain them in detail. Suggest possibilities to improve the aspects, if needed.",
        "formatting":
        """
        Format your response as JSON as follows:
        {
            "discussion_fairness": [
                {{
                    "aspect": string,
                    "explanation": string,
                    "improvement": string
                }},
                ...
            ],
        }
        """
    }

    prompt = f"{USER_PROMPT}{transcript}\n\n"
    prompt += f"{insight['prompt']}\n{insight['formatting']}\n\n"
    return prompt

In [13]:
# Format the prompts to the need OpenAI format
def get_input_messages(system_prompt, user_prompt):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_prompt
                }
            ]
        }
    ]

    return messages

In [14]:
# Generate LLM responses
def get_response(messages):
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        temperature=0.0
    )
    return response

In [15]:
# Submit the analysis step to the LLM by providing the respective prompt. The response is returned as JSON.
def analyze_insight(user_prompt):
    input_messages = get_input_messages(SYSTEM_PROMPT, user_prompt)
    response = get_response(input_messages)
    response_json = json.loads(response.choices[0].message.content.lstrip("```json").rstrip("```"))
    return response_json

## Analysis Workflow
Run the analysis workflow step by step.

The provided transcript sample is a very trivial group discussion to agree on vacation destinations. It was generated with gpt-4o. While it is very simple, it allows to have a look-and-feel regarding the meeting analysis.

In [16]:
# TODO: Replace file
MEETING_NAME = "sample_discussion"

In [17]:
# Load transcript from file
with open(f"{MEETING_NAME}.txt") as f:
    transcript = f.read()

### Identify Aim

In [25]:
aim_result = analyze_insight(aim_prompt(transcript))

In [26]:
def aim_md(result):
    md = f"### Aim\n\n"
    md += f"{result['aim']}\n\nExplanation: {result['explanation']}\n\n"
    return md

In [27]:
print(aim_md(aim_result))

### Aim

The aim of the meeting is to decide on the next vacation destination for the group by discussing and ranking both summer and winter options.

Explanation: The aim is identified through Alice's initial statement where she mentions the need to decide on a vacation destination and suggests collecting and discussing all options before grouping and ranking them. The conversation throughout the meeting revolves around proposing, comparing, and evaluating different vacation destinations for both summer and winter, leading to a final ranking of the options.




### Identify Alternatives

In [22]:
alternatives_result = analyze_insight(alternatives_prompt(transcript))

In [28]:
alternatives = alternatives_result["alternatives"]

In [29]:
def alternatives_md(result):
    md = f"### Identified Alternatives\n\n"
    for alternative in result["alternatives"]:
        md += f"- {alternative}\n"
    return md

In [30]:
print(alternatives_md(alternatives_result))

### Identified Alternatives

- Greece
- Thailand
- Croatia
- Switzerland
- Canada
- Norway



### Recommend Additional Alternatives

In [31]:
recommended_alternatives_result = analyze_insight(recommended_alternatives_prompt(transcript, alternatives))

In [32]:
def recommended_alternatives_md(result):
    md = f"### Recommended Alternatives\n\n"
    for alternative in result["alternatives"]:
        md += f"**{alternative['alternative']}**\n\nExplanation: {alternative['explanation']}\n\n"
    return md

In [33]:
print(recommended_alternatives_md(recommended_alternatives_result))

### Recommended Alternatives

**Italy**

Explanation: Italy offers a mix of cultural experiences, beautiful coastlines, and delicious cuisine. Destinations like the Amalfi Coast or Sicily provide stunning beaches and historical sites, making it a great summer option.

**Spain**

Explanation: Spain is known for its vibrant culture, beautiful beaches, and diverse landscapes. Destinations like Barcelona, Ibiza, or the Canary Islands offer a mix of relaxation, nightlife, and cultural experiences.

**Japan**

Explanation: Japan offers a unique blend of traditional and modern experiences. In winter, destinations like Hokkaido provide excellent skiing opportunities, while in summer, places like Okinawa offer beautiful beaches and cultural experiences.

**New Zealand**

Explanation: New Zealand is ideal for adventure seekers, offering stunning landscapes, hiking, and outdoor activities. It provides both summer and winter experiences, with beaches and ski resorts available.

**Iceland**

Explan

### Analyze Discusssed Arguments per Alternative

In [34]:
single_arguments_result = {}

for alternative in alternatives:
    single_arguments_result[alternative] = analyze_insight(argument_prompt(transcript, alternative))

In [35]:
def single_arguments_md(result):
    md = f"### Arguments about alternatives\n\nArguments in favor are introduced with **+** and against with **-**. Additional arguments provided by the LLM are introduced with **++** and against with **--**.\n\n"
    for alternative, result in result.items():
        md += f"**{alternative}**\n\n"
        for argument in result["arguments_for"]:
            md += f"\+ {argument['argument']}\n\n(Reference: {argument['reference']})\n\n"
        for argument in result["additional_arguments_for"]:
            md += f"\++ {argument['argument']}\n\n"
        for argument in result["arguments_against"]:
            md += f"\- {argument['argument']}\n\n(Reference: {argument['reference']})\n\n"
        for argument in result["additional_arguments_against"]:
            md += f"\-- {argument['argument']}\n\n"
        print("\n")
    return md

In [36]:
print(single_arguments_md(single_arguments_result))













### Arguments about alternatives

Arguments in favor are introduced with **+** and against with **-**. Additional arguments provided by the LLM are introduced with **++** and against with **--**.

**Greece**

\+ Greece offers beautiful destinations like Mykonos and Santorini, with incredible beaches and amazing food.

(Reference: Bob: I’ll start—one of my top choices is Greece. Mykonos and Santorini are beautiful, the beaches are incredible, the food is amazing, and it has a mix of relaxation and nightlife.)

\+ Greece provides a mix of relaxation and nightlife.

(Reference: Bob: I’ll start—one of my top choices is Greece. Mykonos and Santorini are beautiful, the beaches are incredible, the food is amazing, and it has a mix of relaxation and nightlife.)

\+ Greece is a classic Mediterranean beauty destination.

(Reference: Alice: So if we want cost efficiency, Thailand wins. If we want classic Mediterranean beauty, Greece is the choice.)

\++ Greece has a rich historical an

### Identify Discussed Groups

In [37]:
groups_only_result = analyze_insight(groups_only_prompt(transcript, alternatives))

In [38]:
groups = [group["label"] for group in groups_only_result["groups"]]

In [39]:
def groups_md(result):
    md = "### Identified Groups\n\n"
    for group, assignments in result.items():
        md += f"**{group}**:\n\n"
        for alternative in assignments:
            md += f"{alternative}\n\n"
    return md

### Assign Altnernatives to Groups


In [40]:
# Improves assignment
single_groups_result = {}

for alternative in alternatives:
    group_single_result = analyze_insight(groups_assignment_only_single(transcript, alternative, groups))
    if group_single_result["group"] in single_groups_result.keys():
        single_groups_result[group_single_result["group"]].append(alternative)
    else:
        single_groups_result[group_single_result["group"]] = [alternative]

In [41]:
print(groups_md(single_groups_result))

### Identified Groups

**Summer Options**:

Greece

Thailand

Croatia

**Winter Options**:

Switzerland

Canada

Norway




### Identify Group Ranking

In [43]:
# Improves ranking completeness
single_group_ranking_result = {
    "groups": []
}

for group, assignments in single_groups_result.items():
    group_ranking_single_result = analyze_insight(single_group_ranking_prompt(transcript, group, assignments))
    single_group_ranking_result["groups"].append({
        "label": group,
        "rankings": group_ranking_single_result["ranking"]
    })

In [44]:
def group_ranking_md(result, caption="Ranking of Grouped Alternatives"):
    md = f"### **{caption}**\n\n"
    for result in result["groups"]:
        md += f"**{result['label']}**:\n\n"
        for ranking in result["rankings"]:
            md += f"{ranking['rank']}. **{ranking['alternative']}**\n\nExplanation: {ranking['explanation']}\n\n"
        md += "\n"
    return md

In [45]:
print(group_ranking_md(single_group_ranking_result))

### **Ranking of Grouped Alternatives**

**Summer Options**:

1. **Thailand**

Explanation: Thailand is ranked first due to its affordability, variety of activities, and value for money in terms of accommodation, food, and activities.

2. **Greece**

Explanation: Greece is ranked second because of its classic Mediterranean beauty and mix of relaxation and nightlife, but it is more expensive and crowded during peak season compared to Thailand.

3. **Croatia**

Explanation: Croatia is ranked third as it is considered a hidden gem with fewer crowds and a balance of activities, but it offers less variety compared to Thailand.


**Winter Options**:

1. **Norway**

Explanation: Norway is ranked first because it offers a unique winter experience with activities like seeing the Northern Lights, dog sledding, and exploring fjords, which are different from typical skiing experiences.

2. **Switzerland**

Explanation: Switzerland is ranked second as it is considered a dream winter destination wit

### Identify Relationships

In [46]:
relationships_result = analyze_insight(relationships_prompt(transcript, alternatives))

In [47]:
def relationships_md(result):
    md = f"### **Identified Dependencies Between Alternatives**\n\n"
    if len(result["dependencies"]) == 0:
        md += "No dependencies identified.\n\n"
    for alternative in result["dependencies"]:
        md += f"- **{alternative['description']}**\n\n*Explanation:* {alternative['explanation']}\n\n"

    md += f"\n### **Identified Redundancies Between Alternatives**\n\n"
    if len(result["redundancies"]) == 0:
        md += "No inconsistencies identified.\n\n"
    for alternative in result["redundancies"]:
        md += f"- **{alternative['description']}**\n\n*Explanation:* {alternative['explanation']}\n\n"
    return md

In [48]:
print(relationships_md(relationships_result))

### **Identified Dependencies Between Alternatives**

No dependencies identified.


### **Identified Redundancies Between Alternatives**

- **Greece and Croatia**

*Explanation:* Both Greece and Croatia are considered summer destinations with beautiful coastlines, historical towns, and Mediterranean experiences. They offer similar types of activities such as beach relaxation and cultural sightseeing.

- **Switzerland and Canada**

*Explanation:* Both Switzerland and Canada are considered winter destinations with a focus on skiing and other winter activities. They offer similar experiences in terms of skiing resorts and winter sports.




### Evaluate Strategy and Decision Process

In [70]:
prioritization_strategy_result = analyze_insight(prioritization_prompt(transcript))

In [71]:
def prioritization_md(result):
    md = f"### Prioritization Strategy\n\n"
    md += f"{result['prioritization']}\n\nExplanation: {result['explanation']}\n\n"
    return md

In [72]:
print(prioritization_md(prioritization_strategy_result))

### Prioritization Strategy

The group agreed on prioritizing destinations based on cost efficiency, variety of activities, and uniqueness of experience.

Explanation: The group discussed and compared the destinations by evaluating factors such as cost, variety of activities, and uniqueness. For summer destinations, they prioritized Thailand for its affordability and variety, Greece for its classic beauty, and Croatia for its balance and fewer crowds. For winter destinations, they prioritized Norway for its unique experiences, Switzerland for luxury skiing, and Canada for affordability and skiing options. This approach was identified through their discussions and final rankings.




In [60]:
process_evaluation_result = analyze_insight(process_evaluation_prompt(transcript))

In [61]:
def process_evaluation_md(result):
    md = f"### Discussion Process\n\n"
    md += f"Rating: {result['rating']}\n\nExplanation: {result['explanation']}\n\n"
    md += f"Improvement Suggestion: {result['improvement_suggestion']}\n\nImprovement Explanation: {result['improvement_explanation']}\n\n"
    return md

In [62]:
print(process_evaluation_md(process_evaluation_result))

### Discussion Process

Rating: 4

Explanation: The decision-making process was well-structured and inclusive. The group started by listing all potential options without bias towards summer or winter destinations. They then discussed the pros and cons of each option, considering factors like cost, activities, and uniqueness. The group also effectively categorized the options into summer and winter destinations and ranked them based on their preferences and criteria. This approach ensured that all voices were heard and that decisions were made based on a comprehensive analysis of the options.

Improvement Suggestion: Introduce a more formalized decision-making framework, such as a weighted scoring system.

Improvement Explanation: While the group did a good job of discussing and ranking options, a more formalized framework could help quantify their preferences and make the decision-making process more objective. By assigning weights to different criteria (e.g., cost, activities, uniquen

### Evaluate Fairness

In [63]:
discussion_fairness_result = analyze_insight(discussion_fairness_prompt(transcript))

In [64]:
def discussion_fairness_md(result):
    md = f"### Discussion Fairness\n\n"
    for fairness_aspect in result["discussion_fairness"]:
        md += f"**{fairness_aspect['aspect']}**\n\nExplanation: {fairness_aspect['explanation']}\n\nImprovement: {fairness_aspect['improvement']}\n\n"
    return md

In [65]:
print(discussion_fairness_md(discussion_fairness_result))

### Discussion Fairness

**Equal Participation**

Explanation: All participants had the opportunity to contribute their ideas and preferences for both summer and winter vacation options. Each person shared at least one suggestion, and their input was acknowledged.

Improvement: No improvement needed as all participants were actively involved.

**Consideration of Opinions**

Explanation: The group considered all the options presented by each participant. They discussed the pros and cons of each destination, showing that all opinions were taken into account.

Improvement: No improvement needed as all opinions were considered.

**Transparency**

Explanation: The meeting goal was clearly defined at the beginning by Alice, who stated that they needed to decide on a vacation destination. The process of collecting, discussing, and ranking options was also outlined.

Improvement: No improvement needed as the meeting goals and process were transparent.

**Clarification and Extension**

Explanat

## Export Reults to JSON file

In [73]:
import json

response_summary = {}
response_summary["aim"] = aim_result
response_summary["alternatives"] = alternatives_result
response_summary["recommended_alternatives"] = recommended_alternatives_result
response_summary["single_arguments"] = single_arguments_result
response_summary["groups"] = groups_only_result
response_summary["group_ranking"] = single_group_ranking_result
response_summary["relationships"] = relationships_result
response_summary["prioritization_strategy"] = prioritization_strategy_result
response_summary["process_evaluation"] = process_evaluation_result
response_summary["discussion_fairness"] = discussion_fairness_result

In [74]:
import datetime

# Save response to JSON file
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
with open(f"{MEETING_NAME}_report_{timestamp}.json", mode="w") as f:
    f.write(json.dumps(response_summary, indent=4))

## Export Markdown File

In [76]:
# Export LLM evaluation as markdown file
report_md = "# LLM-based Meeting Analysis\n"
report_md += "The information below is derived from the meeting transcript analyzed by the LLM. As the document was generated by an LLM, it may include inaccuracies and should be verified by a human.\n\n"

report_md += aim_md(aim_result)

report_md += single_arguments_md(single_arguments_result)

report_md += recommended_alternatives_md(recommended_alternatives_result)

report_md += group_ranking_md(single_group_ranking_result)

report_md += relationships_md(relationships_result)

report_md += prioritization_md(prioritization_strategy_result)

report_md += process_evaluation_md(process_evaluation_result)

report_md += discussion_fairness_md(discussion_fairness_result)

# Export to markdown file
with open(f"{MEETING_NAME}_report_{timestamp}.md", mode="w") as f:
    f.write(report_md)